In [1]:
import pandas as pd
import duckdb

# Point at our data folder
RAW = "../data/raw"

# Load the small lookup tables — these are tiny, pandas handles them fine
items = pd.read_csv(f"{RAW}/items.csv")
stores = pd.read_csv(f"{RAW}/stores.csv")

print("ITEMS table:", items.shape)
print(items.head())
print("\nColumns:", list(items.columns))

ITEMS table: (4100, 4)
   item_nbr        family  class  perishable
0     96995     GROCERY I   1093           0
1     99197     GROCERY I   1067           0
2    103501      CLEANING   3008           0
3    103520     GROCERY I   1028           0
4    103665  BREAD/BAKERY   2712           1

Columns: ['item_nbr', 'family', 'class', 'perishable']


In [2]:
# How many items are flagged perishable vs not?
print("Perishable flag counts:")
print(items["perishable"].value_counts())

# Which families are perishable, and how many items in each?
perishable_families = (
    items[items["perishable"] == 1]
    .groupby("family")
    .size()
    .sort_values(ascending=False)
)

print("\nPerishable families (item counts):")
print(perishable_families)

print("\nTotal families overall:", items["family"].nunique())

Perishable flag counts:
perishable
0    3114
1     986
Name: count, dtype: int64

Perishable families (item counts):
family
PRODUCE           306
DAIRY             242
BREAD/BAKERY      134
DELI               91
MEATS              84
POULTRY            54
EGGS               41
PREPARED FOODS     26
SEAFOOD             8
dtype: int64

Total families overall: 33


In [3]:
con = duckdb.connect()

query = f"""
SELECT
    i.family,
    COUNT(*) AS row_count,
    COUNT(DISTINCT t.item_nbr) AS n_items,
    COUNT(DISTINCT t.store_nbr) AS n_stores,
    MIN(t.date) AS first_date,
    MAX(t.date) AS last_date
FROM read_csv_auto('{RAW}/train.csv') AS t
JOIN read_csv_auto('{RAW}/items.csv') AS i
    ON t.item_nbr = i.item_nbr
WHERE i.perishable = 1
GROUP BY i.family
ORDER BY row_count DESC
"""

perishable_rows = con.execute(query).df()
print(perishable_rows)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

           family  row_count  n_items  n_stores first_date  last_date
0           DAIRY    8992824      242        54 2013-01-01 2017-08-15
1         PRODUCE    7154303      306        54 2013-03-16 2017-08-15
2    BREAD/BAKERY    4659246      134        54 2013-01-01 2017-08-15
3            DELI    4116396       91        54 2013-01-01 2017-08-15
4           MEATS    2432048       84        54 2013-01-01 2017-08-15
5         POULTRY    1742403       54        54 2013-01-01 2017-08-15
6            EGGS    1580833       41        54 2013-01-01 2017-08-15
7  PREPARED FOODS     766588       26        54 2013-01-01 2017-08-15
8         SEAFOOD     257895        8        54 2013-01-02 2017-08-15


In [4]:
# V1 scope — change this list for V2, nothing else needs to change
FAMILIES = ['MEATS', 'POULTRY', 'DELI', 'PREPARED FOODS', 'SEAFOOD']

# Format the list for the SQL IN clause
family_filter = ", ".join([f"'{f}'" for f in FAMILIES])

extract_query = f"""
SELECT
    t.date,
    t.store_nbr,
    t.item_nbr,
    t.unit_sales,
    t.onpromotion,
    i.family,
    i.class,
    i.perishable,
    s.city,
    s.state,
    s.type   AS store_type,
    s.cluster
FROM read_csv_auto('{RAW}/train.csv') AS t
JOIN read_csv_auto('{RAW}/items.csv') AS i
    ON t.item_nbr = i.item_nbr
JOIN read_csv_auto('{RAW}/stores.csv') AS s
    ON t.store_nbr = s.store_nbr
WHERE i.family IN ({family_filter})
"""

# Run it and write straight to Parquet — never lands fully in RAM
con.execute(f"COPY ({extract_query}) TO '../data/processed/protein_sales.parquet' (FORMAT PARQUET)")

print("Extract complete.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Extract complete.


In [5]:
df = pd.read_parquet('../data/processed/protein_sales.parquet')

print("Shape:", df.shape)
print("\nMemory usage:", round(df.memory_usage(deep=True).sum() / 1024**2, 1), "MB")
print("\n", df.head())
print("\nDtypes:\n", df.dtypes)

Shape: (9315330, 12)

Memory usage: 1368.0 MB

          date  store_nbr  item_nbr  unit_sales onpromotion          family  \
0  2013-01-01         25    108701       1.000         NaN            DELI   
1  2013-01-01         25    159156      29.904         NaN         POULTRY   
2  2013-01-01         25    265266       1.000         NaN  PREPARED FOODS   
3  2013-01-01         25    319093       6.000         NaN            DELI   
4  2013-01-01         25    319094       1.000         NaN            DELI   

   class  perishable     city        state store_type  cluster  
0   2644           1  Salinas  Santa Elena          D        1  
1   2416           1  Salinas  Santa Elena          D        1  
2   2986           1  Salinas  Santa Elena          D        1  
3   2630           1  Salinas  Santa Elena          D        1  
4   2632           1  Salinas  Santa Elena          D        1  

Dtypes:
 date            object
store_nbr        int64
item_nbr         int64
unit_sales    

In [6]:
# 1. Date range and type check
print("Date column type:", df["date"].dtype)
print("Date range:", df["date"].min(), "to", df["date"].max())

# 2. Missing values per column
print("\nMissing values:")
missing = df.isna().sum()
print(missing[missing > 0])

# 3. How much of onpromotion is missing, as a percentage?
pct_missing_promo = df["onpromotion"].isna().mean() * 100
print(f"\nonpromotion missing: {pct_missing_promo:.1f}%")

# 4. Any negative sales? (returns are recorded as negatives in this dataset)
print("\nNegative unit_sales rows:", (df["unit_sales"] < 0).sum())
print("Zero unit_sales rows:", (df["unit_sales"] == 0).sum())

# 5. Cardinality — how many distinct values in key columns
print("\nDistinct counts:")
for col in ["store_nbr", "item_nbr", "family", "city", "state", "store_type", "cluster"]:
    print(f"  {col}: {df[col].nunique()}")

Date column type: object
Date range: 2013-01-01 to 2017-08-15

Missing values:
onpromotion    1981026
dtype: int64

onpromotion missing: 21.3%

Negative unit_sales rows: 429
Zero unit_sales rows: 0

Distinct counts:
  store_nbr: 54
  item_nbr: 263
  family: 5
  city: 22
  state: 16
  store_type: 5
  cluster: 17


In [7]:
# Convert date from string to real datetime
df["date"] = pd.to_datetime(df["date"])
print("Date dtype now:", df["date"].dtype)

# Is onpromotion missing only in the early period?
promo_by_year = df.groupby(df["date"].dt.year)["onpromotion"].apply(
    lambda x: x.isna().mean() * 100
)
print("\n% onpromotion missing, by year:")
print(promo_by_year.round(1))

# Confirm the sparsity math
n_items = df["item_nbr"].nunique()
n_stores = df["store_nbr"].nunique()
n_days = df["date"].nunique()
actual = len(df)
full_grid = n_items * n_stores * n_days

print(f"\nDistinct days: {n_days}")
print(f"Full grid would be: {full_grid:,} rows")
print(f"Actual rows: {actual:,}")
print(f"Grid coverage: {actual / full_grid * 100:.1f}%")

Date dtype now: datetime64[s]

% onpromotion missing, by year:
date
2013    100.0
2014     23.9
2015      0.0
2016      0.0
2017      0.0
Name: onpromotion, dtype: float64

Distinct days: 1684
Full grid would be: 23,916,168 rows
Actual rows: 9,315,330
Grid coverage: 38.9%
